## Module 3: Vibe Mapping & Playlist Recommendation

**Objective:** Wire in the CLIP image encoder to build the full end-to-end pipeline:
image → vibe detection → genre filtering → cosine similarity → playlist.

**Key Operations:**
* **Vibe Taxonomy:** 12 scene categories, each defined by a natural-language probe text and a label.
* **Vibe Detection:** Encode the uploaded image and compare it against all vibe probes to identify the dominant scene type.
* **Genre Filtering:** Children's genres (`children`, `kids`, `disney`) are permanently excluded from the song pool.
* **Recommendation:** Rank the filtered song pool by cosine similarity to the image embedding and return the top-K playlist.

---
Requires outputs from `02_clip_setup.ipynb`:
- `data/song_embeddings.npy` — precomputed song text embeddings `[61670, 512]`
- `data/song_index.csv` — row-aligned song metadata

In [2]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pillow-heif', '--quiet'], check=True)

import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import clip
from PIL import Image
import pillow_heif
pillow_heif.register_heif_opener()   # adds HEIC/HEIF support to PIL

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()

song_embeddings_all = np.load("../data/song_embeddings.npy")
index_df_all        = pd.read_csv("../data/song_index.csv")

assert len(song_embeddings_all) == len(index_df_all), "Embedding matrix and index are misaligned."

# Permanently remove children's genres from the song pool
KIDS_GENRES = "children|kids|disney"
keep_mask       = ~index_df_all["merged_genres"].str.contains(KIDS_GENRES, case=False, na=False)
song_embeddings = song_embeddings_all[keep_mask.values]
index_df        = index_df_all[keep_mask.values].reset_index(drop=True)

print(f"Device              : {device}")
print(f"Total songs loaded  : {len(index_df_all):,}")
print(f"After kids filter   : {len(index_df):,} ({len(index_df_all)-len(index_df):,} removed)")
print(f"Song embeddings     : {song_embeddings.shape}")

python(61321) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


FileNotFoundError: [Errno 2] No such file or directory: '../data/song_embeddings.npy'

---
### 1. Vibe taxonomy

Each vibe is defined by:
- A **label**: short human-readable name shown to the user
- A **probe**: natural-language description that CLIP encodes into a 512-d vector

Vibe detection works by comparing the image embedding to all probe embeddings and picking the closest match.

In [ ]:
VIBE_TAXONOMY = {
    "cozy_cafe": {
        "label": "Cozy Cafe",
        "probe": "a cozy coffee shop with warm lighting and soft background music",
    },
    "winter_snow": {
        "label": "Winter / Snow",
        "probe": "a quiet snowy street on a cold winter evening",
    },
    "beach_summer": {
        "label": "Beach / Summer",
        "probe": "a bright sunny beach in summer with people relaxing",
    },
    "rainy_melancholic": {
        "label": "Rainy / Melancholic",
        "probe": "a rainy night alone in the city feeling nostalgic and sad",
    },
    "party_energetic": {
        "label": "Party / Energetic",
        "probe": "a high-energy dance party with flashing lights and a dancing crowd",
    },
    "romantic_evening": {
        "label": "Romantic Evening",
        "probe": "a candlelit romantic dinner setting at night",
    },
    "nature_peaceful": {
        "label": "Nature / Peaceful",
        "probe": "a peaceful walk through green forests and nature trails",
    },
    "urban_hustle": {
        "label": "Urban / Hustle",
        "probe": "a busy city street at midday with crowds and tall buildings",
    },
    "late_night_drive": {
        "label": "Late Night Drive",
        "probe": "driving alone on a dark highway at night with city lights in the distance",
    },
    "morning_calm": {
        "label": "Morning / Calm",
        "probe": "a calm peaceful morning at home with soft sunlight coming through the window",
    },
    "dark_moody": {
        "label": "Dark / Moody",
        "probe": "a dark moody aesthetic with deep shadows and muted tones",
    },
    "festival_concert": {
        "label": "Festival / Concert",
        "probe": "an outdoor music festival with a large crowd and a stage",
    },
}

vibe_keys   = list(VIBE_TAXONOMY.keys())
vibe_labels = [VIBE_TAXONOMY[k]["label"] for k in vibe_keys]
vibe_probes = [VIBE_TAXONOMY[k]["probe"] for k in vibe_keys]

print(f"{len(VIBE_TAXONOMY)} vibes defined:")
for key, label in zip(vibe_keys, vibe_labels):
    print(f"  {key:22s} → {label}")

12 vibes defined:
  cozy_cafe              → Cozy Cafe
  winter_snow            → Winter / Snow
  beach_summer           → Beach / Summer
  rainy_melancholic      → Rainy / Melancholic
  party_energetic        → Party / Energetic
  romantic_evening       → Romantic Evening
  nature_peaceful        → Nature / Peaceful
  urban_hustle           → Urban / Hustle
  late_night_drive       → Late Night Drive
  morning_calm           → Morning / Calm
  dark_moody             → Dark / Moody
  festival_concert       → Festival / Concert


In [ ]:
# Precompute vibe probe embeddings — shape: [12, 512]
with torch.no_grad():
    tokens = clip.tokenize(vibe_probes, truncate=True).to(device)
    vibe_embeddings = model.encode_text(tokens)
    vibe_embeddings = F.normalize(vibe_embeddings, dim=-1).cpu().numpy()

print(f"Vibe probe embedding matrix: {vibe_embeddings.shape}")

Vibe probe embedding matrix: (12, 512)


---
### 5. Spotify API Integration

The cells below authenticate with the Spotify Web API and add three capabilities:

| Capability | Function | Auth required |
|---|---|---|
| Track lookup (ID, URL, preview) | `spotify_search_track()` | Client credentials |
| Enrich playlist DataFrame | `enrich_playlist_with_spotify()` | Client credentials |
| Push playlist to a user account | `create_spotify_playlist()` | OAuth (user login) |

**Setup:** Create an app at [developer.spotify.com](https://developer.spotify.com/dashboard) and set `SPOTIPY_CLIENT_ID`, `SPOTIPY_CLIENT_SECRET`, and `SPOTIPY_REDIRECT_URI` as environment variables (or hardcode them below — avoid committing secrets to git).


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'spotipy', '--quiet'], check=True)

import spotipy
from spotipy.oauth2 import SpotifyOAuth, SpotifyClientCredentials
import os

SPOTIPY_CLIENT_ID     = os.getenv('SPOTIPY_CLIENT_ID',     'YOUR_CLIENT_ID')
SPOTIPY_CLIENT_SECRET = os.getenv('SPOTIPY_CLIENT_SECRET', 'YOUR_CLIENT_SECRET')
SPOTIPY_REDIRECT_URI  = os.getenv('SPOTIPY_REDIRECT_URI',  'YOUR_REDIRECT_URI')

# Client-credentials flow: read-only search/metadata
sp_cc = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=SPOTIPY_CLIENT_ID,
    client_secret=SPOTIPY_CLIENT_SECRET,
))

# OAuth flow: required for playlist creation on a user account
PLAYLIST_SCOPE = 'playlist-modify-public playlist-modify-private'
sp_oauth = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=SPOTIPY_CLIENT_ID,
    client_secret=SPOTIPY_CLIENT_SECRET,
    redirect_uri=SPOTIPY_REDIRECT_URI,
    scope=PLAYLIST_SCOPE,
))

print('Spotify clients initialised.')
print('  sp_cc    — client-credentials (search / metadata)')
print('  sp_oauth — OAuth (playlist creation)')


python(61362) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Spotify clients initialised.
  sp_cc    — client-credentials (search / metadata)
  sp_oauth — OAuth (playlist creation)


In [4]:
import time

_spotify_cache: dict = {}   # (track_name, artist) -> hit dict | None

def spotify_search_track(track_name: str, artist: str, retries: int = 2) -> dict | None:
    """
    Search Spotify for a track. Returns a dict with keys:
        id, name, artists, album, uri, external_url, preview_url
    Returns None if not found. Results are cached within the session.
    """
    key = (track_name.lower().strip(), artist.lower().strip())
    if key in _spotify_cache:
        return _spotify_cache[key]

    # Strip featured-artist suffixes that trip up search
    clean_name     = track_name.split('(feat')[0].split('(with')[0].strip()
    primary_artist = artist.split(',')[0].split(';')[0].strip()

    query = f'track:"{clean_name}" artist:"{primary_artist}"'
    for attempt in range(retries + 1):
        try:
            results = sp_cc.search(q=query, type='track', limit=1)
            items   = results.get('tracks', {}).get('items', [])
            if items:
                t   = items[0]
                hit = {
                    'id'          : t['id'],
                    'name'        : t['name'],
                    'artists'     : ', '.join(a['name'] for a in t['artists']),
                    'album'       : t['album']['name'],
                    'uri'         : t['uri'],
                    'external_url': t['external_urls'].get('spotify', ''),
                    'preview_url' : t.get('preview_url'),
                }
                _spotify_cache[key] = hit
                return hit
        except spotipy.exceptions.SpotifyException as e:
            if e.http_status == 429:
                time.sleep(int(e.headers.get('Retry-After', 2)) + 1)
            else:
                break
    _spotify_cache[key] = None
    return None


def enrich_playlist_with_spotify(playlist_df: 'pd.DataFrame') -> 'pd.DataFrame':
    """
    Add spotify_id, spotify_url, spotify_uri, preview_url columns to the playlist DataFrame.
    Tracks not found on Spotify receive NaN in those columns.
    """
    ids, urls, uris, previews = [], [], [], []
    for _, row in playlist_df.iterrows():
        hit = spotify_search_track(row['track_name'], row['artists'])
        ids.append(hit['id']          if hit else None)
        urls.append(hit['external_url'] if hit else None)
        uris.append(hit['uri']          if hit else None)
        previews.append(hit['preview_url'] if hit else None)

    df = playlist_df.copy()
    df['spotify_id']  = ids
    df['spotify_url'] = urls
    df['spotify_uri'] = uris
    df['preview_url'] = previews
    return df


def create_spotify_playlist(
    playlist_df: 'pd.DataFrame',
    playlist_name: str,
    description: str = '',
    public: bool = False,
) -> str:
    """
    Create a Spotify playlist from a DataFrame that has a 'spotify_uri' column.
    Requires sp_oauth (OAuth flow). Returns the playlist URL.
    """
    user_id  = sp_oauth.me()['id']
    playlist = sp_oauth.user_playlist_create(
        user_id, playlist_name, public=public, description=description
    )
    uris = playlist_df['spotify_uri'].dropna().tolist()
    for i in range(0, len(uris), 100):
        sp_oauth.playlist_add_items(playlist['id'], uris[i:i + 100])
    url = playlist['external_urls']['spotify']
    print(f'Playlist created: {url}')
    return url


print('Spotify helpers defined: spotify_search_track | enrich_playlist_with_spotify | create_spotify_playlist')


Spotify helpers defined: spotify_search_track | enrich_playlist_with_spotify | create_spotify_playlist


In [5]:
# Quick connectivity test — verify credentials before running the full pipeline
hit = spotify_search_track('Blinding Lights', 'The Weeknd')
if hit:
    print(f"Found : {hit['name']} by {hit['artists']}")
    print(f"  URL : {hit['external_url']}")
    print(f"  URI : {hit['uri']}")
else:
    print('Track not found — check your API credentials.')


Found : Blinding Lights by The Weeknd
  URL : https://open.spotify.com/track/0VjIjW4GlUZAMYd2vXMi3b
  URI : spotify:track:0VjIjW4GlUZAMYd2vXMi3b


---
### 2. Image-to-Playlist Pipeline (with Diversity & Spotify)

The pipeline now runs five steps:

```
1. Image  ──► CLIP image encoder  ──► 512-d image embedding
                                              │
2.                                  cosine sim vs 12 vibe probes  ──► top vibe label
                                              │
3.                              cosine sim vs ~61K song embeddings
                                              │
4.                  diverse_top_k(): artist cap + genre spread filter
                                              │
5.                       Spotify lookup ──► URLs / URIs / optional playlist push
```

**Diversity parameters** (tunable per call):

| Parameter | Default | Effect |
|---|---|---|
| `max_per_artist` | `2` | No artist appears more than N times |
| `min_genre_spread` | `3` | Extends selection until ≥ N distinct genre tokens are covered |

Both are soft constraints — selection stops if the ranked pool is exhausted, and never grows beyond `1.5 × top_k`.


In [6]:
def diverse_top_k(
    sims: 'np.ndarray',
    index_df: 'pd.DataFrame',
    top_k: int = 10,
    max_per_artist: int = 2,
    min_genre_spread: int = 3,
) -> list:
    """
    Return up to top_k song indices satisfying diversity constraints:
      - max_per_artist  : no primary artist contributes more than this many tracks
      - min_genre_spread: extend selection past top_k until this many distinct
                          genre tokens are covered (soft; capped at 1.5 * top_k)
    """
    ranked         = np.argsort(sims)[::-1]
    selected       = []
    artist_counts  = {}
    covered_genres = set()

    for idx in ranked:
        if len(selected) >= top_k:
            if len(covered_genres) >= min_genre_spread:
                break
            if len(selected) >= int(top_k * 1.5):
                break

        row = index_df.iloc[int(idx)]

        # Normalise to primary artist (before comma/feat)
        primary = (
            str(row['artists'])
            .split(',')[0].split(';')[0]
            .split('(feat')[0].split('feat.')[0]
            .strip().lower()
        )
        if artist_counts.get(primary, 0) >= max_per_artist:
            continue

        selected.append(int(idx))
        artist_counts[primary] = artist_counts.get(primary, 0) + 1
        covered_genres |= {
            g.strip().lower()
            for g in str(row.get('merged_genres', '')).split(',')
            if g.strip()
        }

    return selected


def recommend_from_image(
    image_path: str,
    top_k: int = 10,
    max_per_artist: int = 2,
    min_genre_spread: int = 3,
    enrich_spotify: bool = True,
) -> tuple:
    """
    Full pipeline: image -> vibe -> diverse playlist -> (optional) Spotify enrichment.

    Returns (vibe_key, vibe_label, vibe_score, all_vibe_sims, playlist_df)
    """
    # Step 1 — encode image
    try:
        pil_img = Image.open(image_path).convert('RGB')
    except Exception as e:
        raise ValueError(
            f"Could not open '{image_path}'.\n"
            f"iPhone HEIC photos: re-save as JPEG via Share -> Save to Files.\n"
            f"Original error: {e}"
        ) from e

    img_tensor = preprocess(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        img_emb = model.encode_image(img_tensor)
        img_emb = F.normalize(img_emb, dim=-1).cpu().numpy()   # [1, 512]

    # Step 2 — detect vibe
    vibe_sims      = (vibe_embeddings @ img_emb.T).squeeze()   # [12]
    top_vibe_idx   = int(vibe_sims.argmax())
    top_vibe_key   = vibe_keys[top_vibe_idx]
    top_vibe_lbl   = vibe_labels[top_vibe_idx]
    top_vibe_score = float(vibe_sims[top_vibe_idx])

    # Step 3 — cosine similarity vs full song pool
    song_sims = (song_embeddings @ img_emb.T).squeeze()        # [N]

    # Step 4 — diverse selection
    selected_idx = diverse_top_k(
        sims=song_sims,
        index_df=index_df,
        top_k=top_k,
        max_per_artist=max_per_artist,
        min_genre_spread=min_genre_spread,
    )

    rows = []
    for rank, idx in enumerate(selected_idx):
        r = index_df.iloc[idx]
        rows.append({
            'rank'       : rank + 1,
            'similarity' : round(float(song_sims[idx]), 4),
            'track_name' : r['track_name'],
            'artists'    : r['artists'],
            'genres'     : r['merged_genres'],
            'popularity' : int(r['popularity']),
        })
    playlist = pd.DataFrame(rows)

    # Step 5 — optional Spotify enrichment
    if enrich_spotify and not playlist.empty:
        print('Looking up tracks on Spotify...')
        playlist = enrich_playlist_with_spotify(playlist)

    return top_vibe_key, top_vibe_lbl, top_vibe_score, vibe_sims, playlist


def show_result(image_path: str, top_k: int = 10, push_to_spotify: bool = False) -> 'pd.DataFrame':
    """Run the full pipeline and render image + vibe bars + playlist table."""
    vibe_key, vibe_lbl, vibe_score, all_vibe_sims, playlist = recommend_from_image(
        image_path, top_k=top_k
    )

    fig, axes = plt.subplots(1, 2, figsize=(15, 5),
                             gridspec_kw={'width_ratios': [1, 1.6]})
    axes[0].imshow(mpimg.imread(image_path))
    axes[0].axis('off')
    axes[0].set_title(f'Detected vibe: {vibe_lbl}\n(score: {vibe_score:.3f})', fontsize=11)

    sorted_idx  = np.argsort(all_vibe_sims)[::-1]
    sorted_lbls = [vibe_labels[i] for i in sorted_idx]
    sorted_sims = [float(all_vibe_sims[i]) for i in sorted_idx]
    colors = ['tomato' if vibe_labels[i] == vibe_lbl else 'steelblue' for i in sorted_idx]

    axes[1].barh(sorted_lbls[::-1], sorted_sims[::-1], color=colors[::-1])
    axes[1].set_xlim(0.2, 0.85)
    axes[1].set_xlabel('Cosine Similarity')
    axes[1].set_title('Vibe Similarity Scores', fontsize=11)
    plt.tight_layout()
    plt.show()

    print(f'\nTop-{top_k} Playlist — {vibe_lbl} vibe\n')
    display_cols = ['rank', 'similarity', 'track_name', 'artists', 'genres', 'popularity']
    if 'spotify_url' in playlist.columns:
        display_cols.append('spotify_url')
    print(playlist[display_cols].to_string(index=False))

    if push_to_spotify and 'spotify_uri' in playlist.columns:
        import datetime
        create_spotify_playlist(
            playlist,
            playlist_name=f'{vibe_lbl} Vibes — {datetime.date.today()}',
            description=f'Generated by vibe-mapper for a {vibe_lbl} image.',
        )

    return playlist


print('Pipeline functions defined.')
print('  diverse_top_k()      — artist cap + genre-spread selection')
print('  recommend_from_image() — full 5-step pipeline')
print('  show_result()          — visualise + optional Spotify push')


Pipeline functions defined.
  diverse_top_k()      — artist cap + genre-spread selection
  recommend_from_image() — full 5-step pipeline
  show_result()          — visualise + optional Spotify push


---
### 3. Test on sample images

In [7]:
os.makedirs("../data/sample_images", exist_ok=True)

TEST_IMAGES = {
    "Coffee shop" : "../data/sample_images/coffee_shop.jpg",
    "Snowy street": "../data/sample_images/snow_street.jpg",
    "Bright summer": "../data/sample_images/bright_summer.jpg",
    "Party"       : "../data/sample_images/party.jpg",
    "Nature"      : "../data/sample_images/nature.jpg",
}

# Run whichever images exist
for name, path in TEST_IMAGES.items():
    if os.path.exists(path):
        print(f"{'='*60}")
        print(f" Test: {name}")
        print(f"{'='*60}")
        show_result(path, top_k=10)
        print()
    else:
        print(f"[skipped] {name} — file not found at {path}")

 Test: Coffee shop


NameError: name 'vibe_embeddings' is not defined

---
### 4. Vibe detection: all scores

This cell shows the full vibe similarity breakdown for a single image.

In [ ]:
IMAGE_PATH = "../data/sample_images/coffee_shop.jpg"

image = preprocess(Image.open(IMAGE_PATH).convert("RGB")).unsqueeze(0).to(device)
with torch.no_grad():
    img_emb = model.encode_image(image)
    img_emb = F.normalize(img_emb, dim=-1).cpu().numpy()

sims = (vibe_embeddings @ img_emb.T).squeeze()
vibe_score_df = pd.DataFrame({
    "vibe"      : vibe_labels,
    "similarity": sims.round(4),
}).sort_values("similarity", ascending=False).reset_index(drop=True)

print(f"Vibe scores for: {IMAGE_PATH}\n")
print(vibe_score_df.to_string(index=False))

Vibe scores for: ../data/sample_images/coffee_shop.jpg

               vibe  similarity
          Cozy Cafe      0.2812
       Dark / Moody      0.2189
   Romantic Evening      0.2125
  Party / Energetic      0.1946
     Morning / Calm      0.1932
  Nature / Peaceful      0.1784
     Urban / Hustle      0.1648
 Festival / Concert      0.1638
Rainy / Melancholic      0.1633
     Beach / Summer      0.1529
   Late Night Drive      0.1420
      Winter / Snow      0.1358


---
### Summary

The song similarity score measures how close is to the image directly (not to the vibe label).  
The vibe detection and the song ranking are two separate comparisons, both starting from the same image embedding:

```
Image
  │
  ▼
CLIP image encoder → 512-d image embedding
  │
  ├──► vs. 12 vibe probe embeddings  →  top vibe label  (used for display only)
  │
  └──► vs. 61,670 song embeddings    →  similarity score per song  →  top-K playlist
```

- **Vibe similarity** compares the image against 12 short scene descriptions ("a cozy coffee shop with warm lighting..."). This tells you *what kind of scene* CLIP detected in the image, and is shown as the label in the output.

- **Song similarity** compares the same image embedding against each song's `clip_metadata` embedding. This is what actually determines the playlist ranking; each song's score reflects how closely its described mood, genre, and sound match the image's visual content.

The vibe label does not filter or re-rank the songs. It is a human-readable summary of what CLIP saw in the image, added for interpretability. The playlist is ranked purely by cosine similarity between the image and the songs.